# CineSenseAI — 02 Data Cleaning & Normalization

Production data preprocessing: title cleaning, year extraction, genre parsing, and tag aggregation.

*Author: CineSenseAI Team | Dataset: MovieLens Latest Small (GroupLens Research)*

## 1. Title Normalization & Release Year Extraction
MovieLens titles format years at the end (e.g. `Toy Story (1995)`) and often put articles at the end (e.g. `Shawshank Redemption, The`). We extract the integer year and re-format standard titles.

In [ ]:
import re
import pandas as pd
import numpy as np

movies = pd.read_csv('../data/raw/ml-latest-small/movies.csv')

def clean_movie_title(raw_title):
    title = raw_title.strip()
    year = None
    m = re.search(r'\((\d{4})\)$', title)
    if m:
        year = int(m.group(1))
        title = title[:m.start()].strip()
    
    articles = [', The', ', A', ', An', ', Il', ', La']
    for art in articles:
        if title.endswith(art):
            prefix = art.replace(', ', '').strip()
            title = f"{prefix} {title[:-len(art)]}".strip()
            break
    return title, year

res = [clean_movie_title(t) for t in movies['title']]
movies['clean_title'] = [r[0] for r in res]
movies['release_year'] = [r[1] for r in res]

print(movies[['title', 'clean_title', 'release_year']].head())

## 2. Handling Missing Genres and Formatting Lists
`(no genres listed)` must be converted into empty lists rather than false tokens.

In [ ]:
def parse_genres(g_str):
    if not isinstance(g_str, str) or g_str == '(no genres listed)':
        return []
    return [g.strip() for g in g_str.split('|') if g.strip()]

movies['genre_list'] = movies['genres'].apply(parse_genres)
print(f"Movies with '(no genres listed)': {(movies['genres'] == '(no genres listed)').sum()}")

## 3. Aggregating User Tags by Movie
Group individual user tags into clean, lowercased descriptive token sets.

In [ ]:
tags = pd.read_csv('../data/raw/ml-latest-small/tags.csv')
tags_clean = tags.dropna(subset=['tag']).copy()
tags_clean['tag'] = tags_clean['tag'].str.lower().str.strip()

tag_grouped = tags_clean.groupby('movieId')['tag'].apply(lambda s: list(pd.unique(s))).reset_index()
tag_grouped.rename(columns={'tag': 'tags_list'}, inplace=True)

movies = movies.merge(tag_grouped, on='movieId', how='left')
movies['tags_list'] = movies['tags_list'].apply(lambda x: x if isinstance(x, list) else [])
print("Movies with community tags:", (movies['tags_list'].apply(len) > 0).sum())